In [ ]:
from notebook.services.config import ConfigManager
cm = ConfigManager()
cm.update('livereveal', {
        'width': 1920,
        'height': 1080,
        'scroll': True,
})

# Week 09: Monday, AST 5011: Astrophysical Systems

## Density Fluctuations in the Early Universe

### Michael Coughlin

*Reading: CFN Chapter 3*

*Based on notebooks by Benedikt Diemer (U. Maryland) and lectures by Frank van den Bosch (Yale)*

In [ ]:
import numpy as np
import scipy
import matplotlib.pyplot as plt
import matplotlib as mpl
from colossus.cosmology import cosmology
from colossus.utils import constants
from colossus.lss import peaks
import routines

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

cosmo = routines.cosmo
h = cosmo.h

## The Density Field

The matter distribution in the Universe is not perfectly uniform — galaxies, clusters, filaments, and voids trace out the **cosmic web**. To describe this structure quantitatively, we define the **overdensity field**:

$$\delta(\mathbf{x}) \equiv \frac{\rho(\mathbf{x}) - \bar{\rho}}{\bar{\rho}}$$

where $\bar{\rho}$ is the mean density of the Universe. A region with $\delta > 0$ is overdense (more matter than average), and $\delta < 0$ is underdense. The overdensity field is believed to originate from **quantum fluctuations** stretched to macroscopic scales during cosmic inflation.

### Why Fourier Space?

It is natural to decompose the density field in Fourier space:

$$\delta(\mathbf{x}) = \sum_{\mathbf{k}} \delta_{\mathbf{k}} \, e^{i \mathbf{k} \cdot \mathbf{x}}$$

Fourier space is the natural language for describing the density field for several important reasons:
- **Inflation predicts** that different Fourier modes are statistically independent — each mode is drawn from a Gaussian distribution.
- In the **linear regime** ($|\delta| \ll 1$), modes at different $k$ evolve independently, so each wavenumber can be tracked separately.
- The wavenumber $k$ maps directly to a physical scale: $\lambda = 2\pi/k$. Small $k$ corresponds to large scales (superclusters, voids), while large $k$ corresponds to small scales (galaxies, substructure).

The **power spectrum** $P(k)$ quantifies the variance of modes at each scale:

$$P(k) = V \langle |\delta_{\mathbf{k}}|^2 \rangle$$

The dimensionless power spectrum $\Delta^2(k) = k^3 P(k) / (2\pi^2)$ gives the variance per logarithmic interval in $k$. When $\Delta^2(k) \sim 1$, perturbations at that scale are becoming nonlinear — this is where structure formation is actively happening.

In [ ]:
# Visualize how different power spectra affect Fourier mode amplitudes
Nk = 10
Nx = 500
space = 2.5
color_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']

np.random.seed(2024)
phases = np.random.uniform(0.0, 2.0 * np.pi, Nk)
x = np.linspace(0.0, 1.0, Nx)
modes = np.zeros((Nk, Nx), float)

fig, axs = plt.subplots(1, 3, figsize=(12.0, 3.8))
plt.subplots_adjust(wspace=0.1)

for j in range(3):
    for i in range(Nk):
        lam = 1.0 / (i + 1)
        k = 2.0 * np.pi / lam
        if j == 0:
            amp = 10.0 / k
        elif j == 1:
            amp = 1.0
        elif j == 2:
            amp = 0.02 * k
        modes[i] = np.sin(x * k + phases[i]) * amp
    total = np.sum(modes, axis=0)

    plt.sca(axs[j])
    plt.xlim(0.0, 1.0)
    plt.gca().set_xticklabels([])
    plt.gca().set_yticklabels([])
    for i in range(Nk):
        offset = i * space
        plt.axhline(offset, ls='--', color='gray', lw=0.5)
        plt.plot(x, modes[i] + offset, color=color_cycle[0])
    offset = (Nk + 3) * space
    plt.axhline(offset, ls='--', color='gray', lw=0.5)
    plt.plot(x, total + offset, color=color_cycle[1])
    if j == 0:
        label = r'$P \propto k^{-1}$'
    elif j == 1:
        label = r'$P \propto \mathrm{const}$'
    elif j == 2:
        label = r'$P \propto k^{+1}$'
    plt.text(0.05, 0.9, label, transform=plt.gca().transAxes, fontsize=16)
    plt.ylim(-2.0, offset + 10.0)

plt.suptitle('Fourier modes with different power spectrum slopes', y=1.02)
plt.show()

More power at low $k$ (left panel) produces large-scale structure; more power at high $k$ (right panel) produces small-scale structure. Even a flat power spectrum (center) is dominated by small scales because there are more high-$k$ modes in a given volume.

## The Linear Growth Factor

In the linear regime ($|\delta| \ll 1$), each Fourier mode evolves independently — this is the great simplification that makes analytic progress possible. The growth of perturbations is described by the **linear growth factor** $D_+(z)$:

$$\delta(\mathbf{x}, z) = D_+(z) \cdot \delta(\mathbf{x}, z=0)$$

The power spectrum at redshift $z$ is then:

$$P(k, z) = P(k, 0) \cdot D_+^2(z)$$

Notice that $D_+$ depends only on redshift, not on scale — in the linear regime, all modes grow at the same rate. This is why we can separate the spatial pattern (set by initial conditions and the transfer function) from the time evolution.

**Key epochs in the growth history:**
- During the **radiation-dominated era**, perturbations in matter grow only logarithmically (the **Meszaros effect**). Intuitively, the rapid expansion driven by radiation "outruns" gravitational collapse, preventing matter perturbations from growing efficiently.
- After **matter-radiation equality** ($z_\text{eq} \approx 3400$), perturbations grow as $D_+ \propto a$ — the Universe expands slowly enough for gravity to win.
- In the **dark energy-dominated era** ($z \lesssim 0.7$), expansion accelerates again and growth slows, with $D_+$ approaching a constant.

In [ ]:
# Plot the linear growth factor
a = 10**np.linspace(-7.0, 1.5, 100)
z = 1.0 / a - 1.0
D = cosmo._growthFactorExact(z)
print('D+ approaches %.2f in the far future.' % D[-1])

# Exact solution for matter+radiation universe
a_eq = cosmo.a_eq
x = a / a_eq
D_early = a + 2.0 / 3.0 * a_eq + a_eq / (2.0 * np.log(2.0) - 3.0) * \
    (2.0 * np.sqrt(x + 1.0) + (2.0 / 3.0 + x) * np.log((np.sqrt(1.0 + x) - 1.0) / (np.sqrt(1.0 + x) + 1.0)))
D_early *= D[0] / D_early[0]

plt.figure(figsize=(4.5, 4.0))
plt.xlabel(r'$a$')
plt.ylabel(r'$D_+$')
plt.loglog()
plt.xlim(a[0], a[-1])
plt.ylim(2E-4, 50.0)
plt.plot(a, D, lw=1.5, label=r'$\mathrm{Matter + radiation + DE}$')
plt.plot(a, D_early, '--', lw=1.5, label=r'$\mathrm{Matter + radiation}$')
plt.axhline(1.0, ls=':', color='gray', lw=0.7)
plt.axvline(1.0, ls=':', color='gray', lw=0.7)
plt.axvline(a_eq, ls='--', color='gray', lw=0.8, label=r'$a_{\rm eq}$')
plt.legend(frameon=True, labelspacing=0.2)
plt.title('Linear Growth Factor')
plt.show()

## The Jeans Instability

The evolution of density perturbations is governed by a fundamental competition:
- **Gravity** tries to amplify perturbations — an overdense region attracts more matter, becoming even more overdense.
- **Pressure** resists compression — the gas heats up and pushes back against gravitational collapse.

Think of it this way: if you have a small overdense patch, pressure waves (sound) can cross it quickly and smooth it out before gravity has time to act. But if the patch is large enough, the sound-crossing time exceeds the gravitational free-fall time, and gravity wins. The critical dividing line is the **Jeans scale**.

The linearized perturbation equation in Fourier space captures this competition:

$$\frac{d^2 \delta_k}{dt^2} + 2 \frac{\dot{a}}{a} \frac{d\delta_k}{dt} = \left(4\pi G \bar{\rho} - \frac{k^2 c_s^2}{a^2}\right) \delta_k$$

The three terms have clear physical meanings: the left side has the acceleration of the perturbation and a Hubble friction term (expansion dilutes the growth). The right side has gravity (first term, positive, driving growth) versus pressure (second term, negative, resisting growth).

This defines the **Jeans wavenumber** $k_J$:

$$k_J = \frac{a}{c_s}\sqrt{4\pi G \bar{\rho}}$$

- Modes with $k < k_J$ ($\lambda > \lambda_J$): **gravitational collapse** — gravity overwhelms pressure and perturbations grow. These are the seeds of galaxies, clusters, and the cosmic web.
- Modes with $k > k_J$ ($\lambda < \lambda_J$): **pressure support** — pressure is strong enough to resist collapse, and perturbations oscillate as sound waves. These oscillations are the origin of the acoustic peaks in the CMB.

The **Jeans mass** $M_J = (\pi/6)\bar{\rho}\lambda_J^3$ is the minimum mass that can collapse. Before recombination, photons are tightly coupled to baryons, giving a high sound speed ($c_s \approx c/\sqrt{3}$) and a correspondingly large Jeans mass ($M_J \sim 10^{16}\,M_\odot$ — larger than any galaxy cluster!). At recombination ($z \approx 1100$), photons decouple, the sound speed plummets by a factor of $\sim 1000$, and the Jeans mass drops by **$\sim 10$ orders of magnitude** to $M_J \sim 10^5\,M_\odot$. This sudden transition allows baryons to fall into the dark matter potential wells that have been waiting for them.

In [ ]:
# Plot sound speed, Jeans length, and Jeans mass around recombination
z_rec = 1100.0
a_rec = 1.0 / (1.0 + z_rec)
log_a_rec = np.log10(a_rec)
a_arr = 10**np.linspace(log_a_rec - 2.0, log_a_rec + 2.0, 300)
z_arr = 1.0 / a_arr - 1.0
mask_arec = (a_arr > a_rec)
mask_brec = np.logical_not(mask_arec)
cs = np.zeros_like(a_arr)

# Unit conversions
cgs_density = constants.MSUN * h**2 / constants.KPC**3
rho_g = cosmo.rho_gamma(z_arr) * cgs_density
rho_b = cosmo.rho_b(z_arr) * cgs_density
rho_m = cosmo.rho_m(z_arr) * cgs_density
rho_m_0 = cosmo.rho_m(0.0) * cgs_density
rho_tot = rho_g + rho_m

# Sound speed before recombination (photon-baryon fluid)
cs[mask_brec] = constants.C / np.sqrt(3.0) * \
    ((3.0 * rho_b[mask_brec]) / (4.0 * rho_g[mask_brec]) + 1.0)**-0.5

# Sound speed after recombination (ideal gas, T ~ a^-2)
gamma = 5.0 / 3.0
T_bar = cosmo.Tcmb0 / a_rec * (a_arr / a_rec)**-2
P = rho_b * constants.KB * T_bar / (1.22 * constants.M_PROTON)
cs[mask_arec] = np.sqrt(gamma * P[mask_arec] / rho_b[mask_arec])

# Jeans length and mass
l_J = cs * np.sqrt(np.pi / constants.G_CGS / rho_tot) / a_arr
M_J = np.pi / 6.0 * rho_m_0 * l_J**3

fig, axs = plt.subplots(1, 3, figsize=(13.0, 3.4))
plt.subplots_adjust(wspace=0.4)
for i in range(3):
    plt.sca(axs[i])
    plt.xlabel(r'$a$')
    plt.loglog()
    plt.xlim(a_arr[0], a_arr[-1])
    plt.axvline(cosmo.a_eq, ls='--', color='gray', label=r'$a_{\rm eq}$' if i == 0 else None)
    plt.axvline(a_rec, ls='--', color='red', label=r'$a_{\rm rec}$' if i == 0 else None)

plt.sca(axs[0])
plt.ylabel(r'$c_{\rm s} / c$')
plt.plot(a_arr, cs / constants.C)
plt.legend()

plt.sca(axs[1])
plt.ylabel(r'$\lambda_{\rm J,com}\ ({\rm Mpc})$')
plt.plot(a_arr, l_J / constants.MPC)

plt.sca(axs[2])
plt.ylabel(r'$M_{\rm J}\ (M_\odot)$')
plt.plot(a_arr, M_J / constants.MSUN)

plt.suptitle('Sound Speed, Jeans Length, and Jeans Mass around Recombination')
plt.show()

## Exercise 1: The Matter Power Spectrum

Use the Colossus library to explore the matter power spectrum and its evolution with redshift.

**Tasks:**
1. **Compute the matter power spectrum** $P(k)$ at $z = 0$ for wavenumbers $k$ from $10^{-4}$ to $10^{3}\,h/$Mpc using `cosmo.matterPowerSpectrum(k)`.
2. **Compute the dimensionless power spectrum** $\Delta^2(k) = k^3 P(k) / (2\pi^2)$. 
   - *Hint:* You should find that $\Delta^2(k) = 1$ at $k \sim 0.1\text{--}0.3\,h/$Mpc. This is the **nonlinear scale** — perturbations at smaller scales have already gone nonlinear by $z=0$.
3. **Compute the linear growth factor** $D_+(z)$ at $z = 0, 1, 5, 30, 100$ using `cosmo._growthFactorExact(z)`. Normalize so that $D_+(0) = 1$.
4. **Plot $P(k)$ scaled to each redshift** using $P(k, z) = P(k, 0) \cdot D_+^2(z)$. 
   - *Hint:* The curves should be parallel on a log-log plot (same shape, different amplitude). Why is that?
5. **On a separate panel**, mark $k_{\rm eq}$ and add a second x-axis showing the corresponding halo mass.
   - *Hint:* $k_{\rm eq} \approx 0.01\,h/$Mpc. The mass enclosed at scale $k$ is $M \sim (4\pi/3)\bar{\rho}(\pi/k)^3$.

**Physical interpretation questions:**
- What does the **turnover** in $P(k)$ at $k \sim k_{\rm eq}$ represent physically? *(Answer: it separates modes that entered the horizon during radiation domination from those that entered during matter domination.)*
- At what scale does $\Delta^2(k) = 1$ today? What is the corresponding mass scale?
- Why does $P(k) \propto k$ at small $k$ but $P(k) \propto k^{-3}$ at large $k$?

In [ ]:
# Exercise 1: The Matter Power Spectrum

k = 10**np.linspace(-4.0, 3.0, 200)
redshifts = [0, 1, 5, 30, 100]

# 1. Compute P(k) at z=0
Pk_z0 = ... # FILL IN: use cosmo.matterPowerSpectrum(k)

# 2. Compute dimensionless power spectrum
Delta2 = ... # FILL IN: k**3 * Pk_z0 / (2 * np.pi**2)

# 3-4. Compute growth factor and scale P(k) to each redshift
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))

plt.sca(axs[0])
plt.loglog()
plt.xlabel(r'$k\ (h/\mathrm{Mpc})$')
plt.ylabel(r'$P(k)\ (\mathrm{Mpc}^3/h^3)$')

for z in redshifts:
    D_z = ... # FILL IN: cosmo._growthFactorExact(z)
    Pk_z = ... # FILL IN: scale P(k,0) by D^2
    plt.plot(k, Pk_z, label=r'$z = %g$' % z)

plt.legend()
plt.title(r'$P(k)$ at different redshifts')

# 5. Plot Delta^2 with halo mass axis
plt.sca(axs[1])
plt.loglog()
plt.xlabel(r'$k\ (h/\mathrm{Mpc})$')
plt.ylabel(r'$\Delta^2(k)$')
plt.plot(k, Delta2, lw=2)

# Add halo mass axis
logM = np.array([4, 8, 12, 16])
M = 10.0**logM
R = peaks.lagrangianR(M)
k_R = 2.0 * np.pi / R
ax2 = axs[1].twiny()
ax2.set_xscale('log')
ax2.set_xlim(axs[1].get_xlim())
ax2.set_xticks(k_R)
ax2.set_xticklabels([r'$10^{%d}$' % m for m in logM])
ax2.set_xlabel(r'$\mathrm{Halo\ mass}\ (h^{-1}M_\odot)$')

plt.tight_layout()
plt.show()

# Print growth factors
for z in redshifts:
    D_z = cosmo._growthFactorExact(z)
    print(f'D(z={z}) = {D_z:.4f}')

## The Transfer Function

The observed power spectrum at late times can be written as:

$$P(k, t) = P_{\rm initial}(k) \cdot T^2(k) \cdot D_+^2(t)$$

where:
- $P_{\rm initial}(k) \propto k^{n_s}$ is the **primordial (inflationary) power spectrum** with $n_s \approx 0.965$. The fact that $n_s$ is close to (but not exactly) 1 is a key prediction of slow-roll inflation — a perfectly scale-invariant spectrum would have $n_s = 1$ (the Harrison-Zeldovich spectrum).
- $T(k)$ is the **transfer function**, encoding all the complex physics that processes modes between their creation during inflation and the present day: horizon crossing, radiation domination suppression, baryon-photon coupling, and free-streaming.
- $D_+(t)$ is the linear growth factor, capturing the overall amplitude evolution.

**The key physical effect:** Modes that enter the horizon during radiation domination ($k > k_{\rm eq}$) are suppressed because radiation pressure prevents matter from collapsing efficiently. The expansion rate is too fast (set by radiation) for the relatively dilute matter to collapse. This produces the characteristic **turnover** in $P(k)$ at $k_{\rm eq} \approx 0.01\,h/$Mpc, which corresponds to the horizon size at matter-radiation equality.

On large scales ($k \ll k_{\rm eq}$), modes entered the horizon after equality and $T(k) \to 1$: the primordial spectrum is preserved. On small scales ($k \gg k_{\rm eq}$), $T(k) \propto \ln(k)/k^2$, suppressing power and giving the familiar $P(k) \propto k^{-3}$ slope at high $k$.

In [ ]:
# Plot the matter power spectrum, primordial spectrum, and slope
k = 10**np.linspace(-4.0, 3.0, 200)
Pk = cosmo.matterPowerSpectrum(k)
slope = cosmo.matterPowerSpectrum(k, derivative=True)
Delta2 = k**3 * Pk / (2.0 * np.pi**2)

# Primordial spectrum normalized to match at low k
Pk_inflation = k**cosmo.ns
Pk_inflation *= Pk[0] / Pk_inflation[0]
Delta2_inflation = k**3 * Pk_inflation / (2.0 * np.pi**2)

k_eq = 0.014  # h/Mpc (Prada et al. 2018)

fig, axs = plt.subplots(1, 3, figsize=(12.0, 3.5))
plt.subplots_adjust(wspace=0.45, top=0.8)

for i in range(3):
    plt.sca(axs[i])
    plt.xscale('log')
    plt.xlim(k[0], k[-1])
    plt.xlabel(r'$k\ (h/\mathrm{Mpc})$')
    plt.axvline(k_eq, ls=':', color='gray')

plt.sca(axs[0])
plt.yscale('log')
plt.ylabel(r'$P(k)\ (\mathrm{Mpc}^3/h^3)$')
plt.plot(k, Pk, label='LCDM')
plt.plot(k, Pk_inflation, '--', label=r'$P \propto k^{n_s}$')
plt.legend(fontsize=10)

plt.sca(axs[1])
plt.yscale('log')
plt.ylabel(r'$\Delta^2(k)$')
plt.plot(k, Delta2)
plt.plot(k, Delta2_inflation, '--')

plt.sca(axs[2])
plt.ylabel(r'$d\ln P / d\ln k$')
plt.plot(k, slope)
plt.axhline(cosmo.ns, ls='--', color='orange', label=r'$n_s = %.3f$' % cosmo.ns)
plt.legend(fontsize=10)

# Add halo mass axis
logM = np.array([4, 8, 12, 16])
R = peaks.lagrangianR(10.0**logM)
k_R = 2.0 * np.pi / R
for i in range(3):
    ax2 = axs[i].twiny()
    ax2.set_xscale('log')
    ax2.set_xlim(k[0], k[-1])
    ax2.set_xticks(k_R)
    ax2.set_xticklabels([r'$10^{%d}$' % m for m in logM])
    ax2.set_xlabel(r'$\mathrm{Halo\ mass}\ (h^{-1}M_\odot)$', labelpad=12)

plt.show()

### How the Power Spectrum Shapes Large-Scale Structure

The shape of $P(k)$ directly determines the morphology of the cosmic web. Different dark matter models predict different power spectra, leading to dramatically different structures:

- **Cold dark matter (CDM)**: The standard model. CDM particles have negligible thermal velocities, so they preserve power on all scales. This produces a rich hierarchy of structure from the smallest dwarf galaxies ($\sim 10^8\,M_\odot$) to massive galaxy clusters ($\sim 10^{15}\,M_\odot$). Structure forms **bottom-up** (small halos first).
- **Warm dark matter (WDM)**: Particles with keV-scale masses have significant free-streaming velocities in the early Universe, erasing small-scale power below a characteristic cutoff. This produces smoother filaments, fewer dwarf galaxies, and less substructure within halos. The cutoff scale depends on the particle mass.
- **White noise**: Equal power on all scales (no transfer function). This is unphysical but instructive — it shows what structure would look like without the processing by radiation domination.

Below we compare the density fields produced by these models, highlighting how the power spectrum imprints on observable structure.

In [ ]:
# Compare density fields from different power spectrum models
Lbox = 100.0
N = 256
z_ini = 30.0
N_half = int(N / 2)

def Pk_white_noise(k, **kwargs):
    """White noise power spectrum (P = const)."""
    return np.ones_like(k) * 1e3

models = [
    (r'CDM ($\Lambda$CDM)', cosmo.matterPowerSpectrum),
    (r'Warm DM ($m_{\rm WDM} = 0.4$ keV)', routines.powerSpectrumWarmDM),
    (r'White Noise ($P = $ const)', Pk_white_noise),
]

fig, axs = plt.subplots(2, 3, figsize=(13, 8))
plt.subplots_adjust(wspace=0.15, hspace=0.35)

k_plot = 10**np.linspace(-4.0, 3.0, 200)

for j, (label, Pk_func) in enumerate(models):
    np.random.seed(2024)
    k_3d, delta, delta_k, D = routines.gaussianRandomField(Lbox, N, z_ini, Pk_func)
    delta_slice = delta[:, :, N_half]
    max_ext = np.percentile(np.abs(delta_slice), 98)

    # Top row: density field slices
    ax = axs[0, j]
    ax.imshow(delta_slice, cmap='RdBu', vmin=-max_ext, vmax=max_ext,
              extent=(0, Lbox, 0, Lbox))
    ax.set_title(label, fontsize=11)
    ax.tick_params('both', length=0)
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    if j == 0:
        ax.set_ylabel('Density field')

    # Bottom row: power spectra
    ax = axs[1, j]
    Pk_vals = Pk_func(k_plot)
    ax.loglog(k_plot, Pk_vals, lw=2, color='C%d' % j)
    if j > 0:
        Pk_cdm = cosmo.matterPowerSpectrum(k_plot)
        ax.loglog(k_plot, Pk_cdm, '--', lw=1, color='gray', alpha=0.5)
    ax.set_xlabel(r'$k\ (h/\mathrm{Mpc})$')
    if j == 0:
        ax.set_ylabel(r'$P(k)\ (\mathrm{Mpc}^3/h^3)$')
    ax.set_xlim(1e-4, 1e3)

plt.suptitle(r'How the Power Spectrum Shapes Structure ($L = %d$ Mpc/$h$, $z = %d$)' % (int(Lbox), int(z_ini)), y=1.01)
plt.show()

## The Cosmic Microwave Background

The CMB is a snapshot of the Universe at recombination ($z \approx 1100$), when photons decoupled from baryons. Its temperature anisotropies directly probe the density fluctuations at that epoch — making the CMB our most powerful window into the physics of the early Universe.

The CMB temperature field is expanded in spherical harmonics (the natural basis for functions on a sphere, analogous to Fourier modes on a flat surface):

$$\frac{\Delta T}{\bar{T}}(\hat{n}) = \sum_{\ell, m} a_{\ell m} Y_{\ell m}(\theta, \phi)$$

The multipole $\ell$ corresponds to an angular scale $\theta \sim 180°/\ell$. The angular power spectrum $C_\ell = \langle |a_{\ell m}|^2 \rangle$ encodes a wealth of cosmological information:

### What each acoustic peak tells us:

- **First acoustic peak** ($\ell \approx 200$, $\theta \approx 1°$): This is the mode that has completed exactly **one half-oscillation** (maximum compression) by recombination. Its angular position is set by the sound horizon, and is exquisitely sensitive to the **geometry of the Universe**: $\ell \approx 200$ confirms that the Universe is **flat** ($\Omega_{\rm tot} = 1$). A closed universe would shift the peak to smaller $\ell$ (larger angles); an open universe to larger $\ell$.

- **Second acoustic peak** ($\ell \approx 540$): This mode has completed one full oscillation (compression then rarefaction) by recombination. The **ratio of odd to even peaks** is sensitive to the **baryon density** $\Omega_b$. More baryons load the oscillation, enhancing compressions (odd peaks) relative to rarefactions (even peaks). The observed ratio gives $\Omega_b / \Omega_m \approx 0.17$.

- **Third and higher peaks**: Probe increasingly smaller scales and constrain the **matter density** $\Omega_m$, the **dark energy equation of state**, and other parameters. The heights of successive peaks form a pattern that uniquely identifies the cosmological model.

- **Damping tail** ($\ell > 1000$): At small scales, photons can diffuse out of perturbations before recombination, exponentially damping the anisotropies. This **Silk damping** probes the diffusion length and helps constrain $n_s$ and the number of neutrino species.

### CMB Observations

The CMB anisotropies reveal the seed perturbations from which all cosmic structure formed:

![WMAP CMB](figures/1280px-WMAP_image_of_the_CMB_anisotropy.jpg)

These perturbations are at the level of $\delta T / T \sim 10^{-5}$ — deep in the linear regime. Yet by today, they have grown into the cosmic web of galaxies, filaments, and voids:

![SDSS Large-Scale Structure](figures/Slices-through-the-SDSS-3-dimensional-map-of-the-distribution-of-galaxies-Earth-is-at.png)

### Acoustic Oscillations and BAO

Before recombination, photons and baryons form a tightly coupled fluid. Gravity pulls this fluid into dark matter overdensities, but radiation pressure resists compression, setting up **acoustic oscillations** — sound waves propagating through the primordial plasma.

The physical picture: imagine each initial overdensity launching a spherical sound wave into the baryon-photon fluid. This wave propagates outward at $c_s \approx c/\sqrt{3}$ until recombination, at which point it has traveled a distance equal to the **sound horizon** ($r_s \sim 150$ Mpc). At recombination, photons decouple, the pressure disappears, and the wave freezes in place.

This characteristic scale is imprinted in:
- **The CMB angular power spectrum**: The acoustic peaks at $\ell \approx 200, 540, 800, \ldots$ correspond to modes that have completed $1/2, 1, 3/2, \ldots$ oscillations by recombination.
- **The galaxy correlation function at late times**: Baryons that were carried outward by the sound wave eventually fall into dark matter halos, creating a statistical excess of galaxy pairs separated by $\sim 150$ Mpc. This is the **baryon acoustic oscillation (BAO)** signal.

BAO serve as a "standard ruler" for measuring cosmic distances — the physical size of the sound horizon is known from CMB physics, so measuring its apparent angular (or redshift-space) size at different epochs constrains the expansion history and dark energy.

### The Matter Correlation Function and the BAO Feature

The **two-point correlation function** $\xi(r)$ is the Fourier transform of the power spectrum:

$$\xi(r) = \int \frac{dk}{k} \frac{k^3 P(k)}{2\pi^2} \frac{\sin kr}{kr}$$

Physically, $\xi(r)$ measures the *excess probability* (relative to a random distribution) of finding two galaxies separated by distance $r$. If $\xi(r) > 0$, galaxies are more clustered than random at that separation; if $\xi(r) < 0$, they are less clustered (anti-correlated).

The BAO feature appears as a bump in $\xi(r)$ at $r \approx 105\,\mathrm{Mpc}/h$ (or $\sim 150$ Mpc in physical units), corresponding to the comoving sound horizon at recombination. This feature has been detected in galaxy surveys (SDSS, BOSS, DESI) and serves as a **standard ruler** for measuring cosmological distances at different redshifts.

In [ ]:
# Compute the matter correlation function using colossus
R_cf = 10**np.linspace(-0.5, 2.5, 300)  # Mpc/h
xi = cosmo.correlationFunction(R_cf)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: xi(r) on log-log scale
ax = axes[0]
mask_pos = xi > 0
ax.loglog(R_cf[mask_pos], xi[mask_pos], lw=2, color='C0')
mask_neg = xi < 0
ax.loglog(R_cf[mask_neg], -xi[mask_neg], '--', lw=1.5, color='C0', alpha=0.5)
ax.set_xlabel(r'$r\ (\mathrm{Mpc}/h)$')
ax.set_ylabel(r'$\xi(r)$')
ax.set_title('Matter Correlation Function')
ax.axvline(105.0, ls=':', color='gray', lw=1, label=r'BAO scale $\sim 105$ Mpc/$h$')
ax.legend(fontsize=9)

# Right: r^2 * xi(r) to highlight the BAO bump
ax = axes[1]
ax.plot(R_cf, R_cf**2 * xi, lw=2, color='C1')
ax.axhline(0.0, ls='-', color='gray', lw=0.5)
ax.axvline(105.0, ls=':', color='gray', lw=1)
ax.set_xlabel(r'$r\ (\mathrm{Mpc}/h)$')
ax.set_ylabel(r'$r^2 \xi(r)\ (\mathrm{Mpc}/h)^2$')
ax.set_title(r'BAO Feature in $r^2\xi(r)$')
ax.set_xlim(0, 200)
ax.text(110, ax.get_ylim()[1] * 0.8, 'BAO\nbump', fontsize=10, color='gray')

plt.tight_layout()
plt.show()

# Find BAO peak location
mask_bao = (R_cf > 80) & (R_cf < 150)
r2xi = R_cf**2 * xi
idx_peak = np.argmax(r2xi[mask_bao])
r_bao = R_cf[mask_bao][idx_peak]
print(f'BAO peak in r^2 xi(r) at r = {r_bao:.1f} Mpc/h')

## Gaussian Random Fields

The density field from inflation is a **Gaussian random field**: its statistical properties are completely specified by the power spectrum $P(k)$. This is a remarkably powerful statement — it means that all the higher-order statistics (three-point function, four-point function, etc.) are either zero or determined by $P(k)$.

**Why Gaussian?** The central limit theorem provides intuition: if the density at each point is the sum of many independent random contributions (as in multi-field inflation or any process with many degrees of freedom), the result is Gaussian. Single-field slow-roll inflation produces Gaussianity for a deeper reason: the perturbations are free-field quantum fluctuations, which are inherently Gaussian.

To create a Gaussian random field numerically:
1. Generate 3D white noise (Gaussian with zero mean, unit variance at each grid point)
2. Fourier transform to get $\delta_k$ (which is also Gaussian, by linearity of the FFT)
3. Weight each mode by $\sqrt{P(k)} \cdot D_+(z)$ — this imprints the desired power spectrum
4. Inverse FFT to get $\delta(\mathbf{x})$

The `gaussianRandomField()` function in `routines.py` implements this procedure. Note that this generates the **linear** density field — it does not include nonlinear gravitational evolution, which would require an N-body simulation.

In [ ]:
# Generate and plot a Gaussian random field
Lbox = 100.0  # Mpc/h
N = 128
z_ini = 30.0
N_half = int(N / 2)

np.random.seed(2024)
k, delta, delta_k, D = routines.gaussianRandomField(Lbox, N, z_ini, cosmo.matterPowerSpectrum)

delta_slice = delta[:, :, N_half]
max_ext = max(np.max(delta_slice), np.abs(np.min(delta_slice))) * 0.8

plt.figure(figsize=(5.0, 5.0))
plt.xlabel(r'$x\ (\mathrm{Mpc}/h)$')
plt.ylabel(r'$y\ (\mathrm{Mpc}/h)$')
plt.gca().tick_params('both', length=0)
im = plt.imshow(delta_slice, cmap='RdBu', vmin=-max_ext, vmax=max_ext,
                extent=(0.0, Lbox, 0.0, Lbox))
cbar = plt.colorbar(im, shrink=0.8, pad=0.05, aspect=10)
cbar.set_label(r'$\delta$')
plt.title(r'Gaussian Random Field ($L = %d$ Mpc/$h$, $z = %d$)' % (int(Lbox), int(z_ini)))
plt.show()

## Exercise 2: Generating and Analyzing a Gaussian Random Field

In this exercise, you will generate a Gaussian random field and verify that its power spectrum matches the input model. This is the fundamental technique behind creating initial conditions for cosmological simulations.

**Tasks:**
1. **Generate a 3D Gaussian random field** in a box of size $L = 100$ Mpc/$h$ with $N = 128$ grid cells per side, using the procedure described above (or the provided `gaussianRandomField()` function).
2. **Plot a 2D slice** through the midplane of the box. 
   - *Hint:* Overdense regions should appear as peaks (bright spots). The typical fluctuation amplitude should be $|\delta| \sim 0.1\text{--}1$ depending on the redshift you choose.
3. **Measure the power spectrum** $P(k)$ from the realization by binning $|\delta_k|^2$ in spherical $k$-shells.
   - *Step-by-step:* Compute $\delta_k$ via FFT, calculate $|\delta_k|^2$ for each mode, bin modes by $|k|$, and average within each bin. Multiply by $V = L^3$ to get $P(k)$.
4. **Compare the measured $P(k)$** to the input LCDM model from Colossus.
   - *Hint:* The measured spectrum should scatter around the input model. At low $k$ (few modes per bin), you will see large **cosmic variance** — the scatter is $\sim 1/\sqrt{N_{\rm modes}}$ per bin. At high $k$, many modes are averaged and the agreement should be tight.

**Physical interpretation:**
- Why is there more scatter at low $k$? What does this imply for measuring the power spectrum from real galaxy surveys on the largest scales?
- If you regenerated the field with a different random seed, the density map would look completely different — but the power spectrum would be statistically the same. Why?

In [ ]:
# Exercise 2: Generating and Analyzing a Gaussian Random Field

Lbox = 100.0  # Mpc/h
N = 128
z_ini = 30.0
N_k_bins = 20
N3 = N**3
N_half = int(N / 2)
np.random.seed(42)

# 1. Generate the field
k, delta, delta_k, D = ... # FILL IN: call routines.gaussianRandomField(Lbox, N, z_ini, cosmo.matterPowerSpectrum)

# 2. Plot a 2D slice
delta_slice = ... # FILL IN: extract the midplane slice delta[:, :, N_half]

plt.figure(figsize=(5, 5))
max_ext = np.percentile(np.abs(delta_slice), 95)
plt.imshow(delta_slice, cmap='RdBu', vmin=-max_ext, vmax=max_ext,
           extent=(0, Lbox, 0, Lbox))
plt.colorbar(label=r'$\delta$', shrink=0.8)
plt.xlabel(r'$x\ (\mathrm{Mpc}/h)$')
plt.ylabel(r'$y\ (\mathrm{Mpc}/h)$')
plt.title('Density Field Slice')
plt.show()

# 3. Measure P(k) from the realization
log_k = np.log10(k)
D2 = D**2

# Bin |delta_k|^2 by log(k) to get P(k)
Pk_measured, bin_edges = ... # FILL IN: np.histogram(log_k, bins=N_k_bins, weights=np.abs(delta_k)**2)
N_modes, _ = ... # FILL IN: np.histogram(log_k, bins=N_k_bins)
Pk_measured = ... # FILL IN: Pk_measured / N_modes / D2 / N3

# Compute average k in each bin
log_k_avg, _ = np.histogram(log_k, bins=N_k_bins, weights=log_k)
log_k_avg /= N_modes
k_avg = 10.0**log_k_avg

# 4. Compare to input model
k_model = 10**np.linspace(-2.0, 2.0, 200)
Pk_model = cosmo.matterPowerSpectrum(k_model)

plt.figure(figsize=(5, 4))
plt.loglog()
plt.xlabel(r'$k\ (h/\mathrm{Mpc})$')
plt.ylabel(r'$P(k)\ (h^{-3}\,\mathrm{Mpc}^3)$')
plt.xlim(1E-2, 1E2)
plt.ylim(1E-3, 1E5)
plt.plot(k_model, Pk_model, lw=2, label='Input model')
plt.plot(k_avg, Pk_measured, 'o', markersize=4, label='Measured from realization')
plt.legend(loc=3)
plt.title('Power Spectrum: Model vs. Realization')
plt.show()

## Smoothing the Density Field

We often want to know the average overdensity on a particular scale $R$. For example: "what is the typical overdensity in a sphere of radius 8 Mpc/$h$?" This is done by convolving the density field with a **smoothing filter** $W(r; R)$.

In Fourier space, convolution becomes a simple multiplication:

$$\delta_R(\mathbf{x}) = \mathcal{F}^{-1}\left[\delta_k \cdot \widetilde{W}(kR)\right]$$

Common filters:
- **Tophat** (sharp in real space): $\widetilde{W}(kR) = 3(\sin kR - kR\cos kR)/(kR)^3$. This corresponds to averaging over a sphere of radius $R$ — the most physically intuitive choice, and the standard for defining $\sigma(R)$ and the halo mass function.
- **Gaussian** (smooth in both spaces): $\widetilde{W}(kR) = e^{-k^2 R^2 / 2}$. Mathematically convenient and avoids the ringing artifacts of the tophat filter.

The **variance** of the smoothed field is:

$$\sigma^2(R) = \int \frac{dk}{k} \frac{k^3 P(k)}{2\pi^2} |\widetilde{W}(kR)|^2$$

This tells us the typical amplitude of fluctuations on scale $R$. When $\sigma(R) \gtrsim 1$, perturbations at that scale are nonlinear and collapsing to form bound structures.

The parameter $\sigma_8 \equiv \sigma(R = 8\,\mathrm{Mpc}/h) \approx 0.81$ is one of the six fundamental cosmological parameters of $\Lambda$CDM. The choice of $R = 8$ Mpc/$h$ is historical — it corresponds roughly to the scale of galaxy clusters, where the transition from linear to nonlinear happens today.

In [ ]:
# Plot smoothing filter functions in Fourier space
kR = np.linspace(1E-5, 20.0, 200)

plt.figure(figsize=(4.5, 3.5))
plt.xlabel(r'$kR$')
plt.ylabel(r'$\widetilde{W}(kR)$')
plt.xlim(0.0, kR[-1])
plt.plot(kR, routines.filterFourierSpace(kR, filt='tophat'), label='Tophat')
plt.plot(kR, routines.filterFourierSpace(kR, filt='gaussian'), label='Gaussian')
plt.legend()
plt.title('Smoothing Filters in Fourier Space')
plt.show()

In [ ]:
# Show smoothed density fields at different scales
Lbox = 100.0
N = 256
z_ini = 30.0
N_half = int(N / 2)

np.random.seed(2024)
k_3d, delta, delta_k_3d, D = routines.gaussianRandomField(Lbox, N, z_ini, cosmo.matterPowerSpectrum)

Rf_array = [0.0, 1, 3, 8]
max_ext = 0.1

fig, axs = plt.subplots(1, 4, figsize=(14.0, 4.0))
plt.subplots_adjust(wspace=0.1)

for i, Rf in enumerate(Rf_array):
    if Rf > 0.0:
        kR = k_3d * Rf
        delta_smooth = np.fft.fftn(delta) * routines.filterFourierSpace(kR, 'tophat')
        delta_smooth = np.real(np.fft.ifftn(delta_smooth))
        label = r'$R = %d$ Mpc/$h$' % Rf
    else:
        delta_smooth = delta
        label = 'No smoothing'

    ax = axs[i]
    plt.sca(ax)
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.tick_params('both', length=0)
    plt.imshow(delta_smooth[:, :, N_half], cmap='RdBu', vmin=-max_ext, vmax=max_ext,
               extent=(0, Lbox, 0, Lbox))
    plt.text(0.05, 0.89, label, transform=ax.transAxes, fontsize=11,
             bbox=dict(fc='w', lw=0))
    if Rf > 0:
        ax.add_artist(mpl.patches.Circle((Lbox * 0.9, Lbox * 0.9), Rf,
                                          color='k', fill=False))

plt.suptitle('Smoothed Density Fields (Tophat Filter)')
plt.show()

## Exercise 3: Smoothing and Cosmic Variance

In this exercise, you will smooth the density field at different scales and compute the variance $\sigma(R)$. This connects directly to the question: "at what mass scale is the Universe going nonlinear today?"

**Tasks:**
1. **Generate a Gaussian random field** using the provided settings ($L = 100$ Mpc/$h$, $N = 128$).
2. **Smooth the field at $R = 1, 5, 10$ Mpc/$h$** using the tophat filter in Fourier space.
   - *Step-by-step:* Multiply $\delta_k$ by $\widetilde{W}_{\rm tophat}(kR)$, then inverse FFT.
   - *Hint:* The $R = 1$ Mpc/$h$ slice should look similar to the unsmoothed field (most power is on larger scales). The $R = 10$ Mpc/$h$ slice should be noticeably smoother, with only the largest structures visible.
3. **Plot the three smoothed slices** side by side. Observe how increasing $R$ erases small-scale structure.
4. **Compute $\sigma(R)$** using `cosmo.sigma(R, z=0)` from Colossus, for $R$ from $10^{-1}$ to $10^2$ Mpc/$h$.
   - *Hint:* You should find a monotonically decreasing function: $\sigma$ is large at small $R$ (small scales are more nonlinear) and small at large $R$ (large scales are still linear).
5. **Identify $\sigma_8$** on your plot and compare to the Planck cosmological parameter value.
   - *Expected result:* You should find $\sigma_8 \approx 0.81$ for the Planck 2018 cosmology.

**Physical interpretation:**
- At what scale $R$ does $\sigma(R) = 1$? What is the corresponding mass? This is the **nonlinear mass scale** today.
- Why does $\sigma(R)$ decrease with increasing $R$? What does this imply about the relative importance of large-scale vs. small-scale perturbations?
- How would $\sigma(R)$ change if we evaluated it at $z = 1$ instead of $z = 0$?

In [ ]:
# Exercise 3: Smoothing and Cosmic Variance

Lbox = 100.0
N = 128
z_ini = 30.0
N_half = int(N / 2)
np.random.seed(2024)

# 1. Generate field (this one is given to you)
k_3d, delta, delta_k_3d, D = routines.gaussianRandomField(Lbox, N, z_ini, cosmo.matterPowerSpectrum)
delta_k_full = np.fft.fftn(delta)

# 2-3. Smooth at three scales and plot
Rf_list = [1, 5, 10]  # Mpc/h
fig, axs = plt.subplots(1, 3, figsize=(12, 4))
plt.subplots_adjust(wspace=0.1)

for i, Rf in enumerate(Rf_list):
    kR = ... # FILL IN: k_3d * Rf
    W = ... # FILL IN: routines.filterFourierSpace(kR, 'tophat')
    delta_smooth = ... # FILL IN: np.real(np.fft.ifftn(delta_k_full * W))

    ax = axs[i]
    plt.sca(ax)
    vmax = np.percentile(np.abs(delta_smooth[:, :, N_half]), 95)
    plt.imshow(delta_smooth[:, :, N_half], cmap='RdBu', vmin=-vmax, vmax=vmax,
               extent=(0, Lbox, 0, Lbox))
    plt.title(r'$R = %d$ Mpc/$h$' % Rf)
    ax.set_xticklabels([])
    ax.set_yticklabels([])

plt.suptitle('Tophat-Smoothed Density Fields')
plt.show()

# 4-5. Compute sigma(R) using Colossus
R_plot = 10**np.linspace(-1, 2, 100)  # Mpc/h

sigma_R = ... # FILL IN: cosmo.sigma(R_plot)
sigma_8 = ... # FILL IN: np.interp(8.0, R_plot, sigma_R)

plt.figure(figsize=(5, 4))
plt.loglog()
plt.xlabel(r'$R\ (\mathrm{Mpc}/h)$')
plt.ylabel(r'$\sigma(R)$')
plt.plot(R_plot, sigma_R, lw=2)
plt.axvline(8.0, ls='--', color='gray', label=r'$R = 8$ Mpc/$h$')
plt.axhline(sigma_8, ls='--', color='red', alpha=0.5)
plt.legend()
plt.title(r'Cosmic Variance: $\sigma_8 = %.3f$ (Colossus: %.3f)' % (sigma_8, cosmo.sigma8))
plt.show()

print(f'sigma_8 from interpolation: {sigma_8:.4f}')
print(f'sigma_8 from cosmology:     {cosmo.sigma8:.4f}')

## From Linear Perturbations to Halo Formation

Everything we have discussed so far applies in the **linear regime** ($\delta \ll 1$). But the most interesting objects in the Universe — galaxies, clusters, stars — live in the deeply nonlinear regime ($\delta \gg 1$). How do we bridge this gap?

The linear growth factor $D_+(z)$ tells us how perturbations grow in time. When the *linearly extrapolated* overdensity reaches a critical threshold $\delta_c \approx 1.686$ (derived from the spherical collapse model), the perturbation has actually already decoupled from the Hubble flow and collapsed to form a **dark matter halo**.

The **non-linear mass** $M_{\rm nl}(z)$ is the mass scale at which the typical perturbation ($\nu = 1$, i.e., $\sigma(M) = \delta_c$) is just collapsing at redshift $z$. This defines the "knee" of the halo mass function — the characteristic mass of halos forming at each epoch.

The **Press-Schechter** formalism (1974) predicts the number density of halos as a function of mass and redshift:

$$\frac{dn}{d\ln M} = \frac{\bar{\rho}}{M} f(\nu) \left|\frac{d\ln\sigma}{d\ln M}\right|$$

where $\nu = \delta_c / \sigma(M, z)$ is the **peak height** — the number of standard deviations a region must fluctuate to collapse. Rare massive clusters have $\nu \gg 1$ (far out on the Gaussian tail), while common dwarf halos have $\nu \ll 1$.

Structure forms **bottom-up** in CDM cosmology: low-mass halos collapse first (because $\sigma$ is larger on small scales), and progressively larger halos form at later times through mergers and accretion. This hierarchical assembly is one of the defining predictions of the CDM paradigm.

In [ ]:
from colossus.lss import mass_function

# Left panel: Non-linear mass evolution (mass scale collapsing at each epoch)
a_arr = 10**np.linspace(-2.0, 0.0, 100)
z_arr = 1.0 / a_arr - 1.0
M_nl = np.array([peaks.nonLinearMass(z) for z in z_arr]) / h  # Msun

# Non-linear mass for different peak heights
nu_values = [0.5, 1.0, 2.0, 4.0]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
for nu in nu_values:
    # sigma(M, z) = delta_c / nu  =>  need to find M where sigma(M) * D(z) = delta_c / nu
    M_nu = []
    for z in z_arr:
        D = cosmo.growthFactor(z)
        sigma_target = 1.686 / (nu * D)
        # Find M where sigma(R(M)) = sigma_target
        logM_test = np.linspace(4, 18, 500)
        R_test = peaks.lagrangianR(10**logM_test * h)  # needs Msun/h
        sig_test = cosmo.sigma(R_test, z=0.0)
        idx = np.argmin(np.abs(sig_test - sigma_target))
        M_nu.append(10**logM_test[idx])
    ax.semilogy(a_arr, M_nu, lw=2, label=r'$\nu = %.1f$' % nu)

ax.set_xlabel(r'Scale factor $a$')
ax.set_ylabel(r'$M\ (M_\odot)$')
ax.set_title('Mass Scales Collapsing at Each Epoch')
ax.set_xlim(a_arr[0], a_arr[-1])
ax.legend(fontsize=9)
ax.axhline(1e12, ls=':', color='gray', lw=0.7)

# Right panel: Halo mass function at different redshifts
ax = axes[1]
logM_arr = np.linspace(8, 16, 100)
M_arr = 10**logM_arr

for z in [0, 1, 2, 4, 7]:
    mfunc = mass_function.massFunction(M_arr * h, z, mdef='200m', model='press74', q_out='dndlnM')
    mfunc *= h**3  # convert from (Mpc/h)^-3 to Mpc^-3
    ax.plot(logM_arr, np.log10(mfunc), lw=2, label=r'$z = %d$' % z)

ax.set_xlabel(r'$\log_{10}(M / M_\odot)$')
ax.set_ylabel(r'$\log_{10}(dn/d\ln M)\ [\mathrm{Mpc}^{-3}]$')
ax.set_title('Press-Schechter Halo Mass Function')
ax.set_xlim(8, 16)
ax.set_ylim(-12, 2)
ax.legend(fontsize=9, ncol=2)

plt.tight_layout()
plt.show()

M_nl_today = peaks.nonLinearMass(0.0) / h
print(f'Non-linear mass today (nu=1): M_nl = {M_nl_today:.2e} Msun')
print(f'This is comparable to the Milky Way halo mass!')

## Summary

### The story of structure formation, in six acts:

1. **The density field and its power spectrum**: The matter distribution is described statistically by $P(k)$, which traces back to quantum fluctuations during inflation. The overdensity $\delta(\mathbf{x})$ is most naturally described in Fourier space, where different modes are independent in the linear regime and the power spectrum fully characterizes the Gaussian field.

2. **Linear growth**: Perturbations grow as $\delta \propto D_+(z)$, with the growth rate depending on the cosmic epoch. Growth is suppressed during radiation domination (**Meszaros effect**) and during dark energy domination. The shape of $P(k)$ is preserved during linear evolution — only the amplitude changes.

3. **Jeans instability**: The competition between gravity and pressure defines the Jeans scale. Perturbations larger than $\lambda_J$ collapse; smaller ones oscillate as sound waves. At recombination, the sudden decoupling of photons from baryons drops $M_J$ by $\sim 10$ orders of magnitude, allowing baryons to fall into dark matter potential wells.

4. **The transfer function and the shape of $P(k)$**: Modes entering the horizon during radiation domination are suppressed, producing the characteristic turnover at $k_{\rm eq} \approx 0.01\,h/$Mpc. This is why $P(k) \propto k$ on large scales but $P(k) \propto k^{-3}$ on small scales. Different dark matter models (CDM, WDM) predict different transfer functions and hence different small-scale structure.

5. **The CMB and acoustic oscillations**: The photon-baryon fluid before recombination undergoes acoustic oscillations, imprinting a series of peaks in the CMB angular power spectrum. Each peak encodes different cosmological information (geometry, baryon density, matter density). The same sound horizon scale appears as the BAO feature in the galaxy correlation function — a standard ruler for measuring cosmic distances and constraining dark energy.

6. **From $P(k)$ to halos**: Smoothing the density field at scale $R$ gives $\sigma(R)$, with $\sigma_8 \approx 0.81$. When $\sigma(M) \cdot D_+(z) \geq \delta_c \approx 1.686$, regions of mass $M$ collapse to form dark matter halos. Structure forms **bottom-up**: small halos first, large halos later. The Press-Schechter formalism connects the linear power spectrum to the abundance of halos as a function of mass and redshift.